# Deep Network Development, 2026 Autumn - Homework assignment

In the homework assignment, you need to train and evaluate a simple Multi-layer Perceptron (MLP) neural network on a regression task and then, a classification task using the PyTorch library.

The homework consists of several subtasks (A - I) that build on one another. The assignment includes an automatic testing script that checks the correctness of your solutions for the individual subtasks. Therefore, **do not change the structure of the notebook**, do not split it into multiple parts, and do not modify the pre-written sections. Only enter your solutions in the appropriate places, otherwise we will not be able to evaluate your work. Your solutions must be written between the `# Implement your solution BELOW` and `# Implement your solution ABOVE` lines. If the message "Tester: .... OK" appears when you run the solved subtasks, the solution is _possibly_ correct. However, the testing script cannot verify correctness in every aspect. Therefore, it is possible that after submission we may ask you to make additional corrections to your solution.

**!!! It is forbidden to publish the solutions to the tasks !!!**

If you get stuck, it is recommended to review the supporting notebooks and videos shared on Moodle, as well as the lecture materials. We collect frequently asked questions and answers on the Moodle forum under the title "Homework assignment - Frequently Asked Questions". You may also ask your practice leader for help.

**Solution and submission procedure:**

It is recommended to open and solve this notebook on your own computer in a Python environment with Jupyter Lab, since you will also need to work in such an environment during the Coding test. More information about setting up this environment can be found on Moodle in the video "Development Environment Setup Guide (Video)" and in the document "Development Environment Setup Guide (PDF)" located in the "Week 1" module. The packages in the university computer labs have been updated since the video and the document was created, but the setup procedure remains the same with the exception of the exact package versions. These are published in the `requirements.txt` file, which is available for download next to the guide video and document on Moodle, under the name "requirements.txt file for the Coding test environment".

The notebook can be solved using other development environments (e.g., online with Google Colab, or locally with Visual Studio Code), however, these will not be accessible during the Coding test, so it is recommended to practice with Jupyter Lab.

The completed notebook containing your solution must be uploaded to Moodle as an `.ipynb` file via the homework submission page. The other files (the tester script and the dataset .csv) should not be submitted.
**Please do not modify the notebook structure and do not split it into multiple notebooks by task.**

**Deadline:** Thursday, 19 November, 2026, 23:59.

**Late submission deadline:** Thursday, 26 November, 2026, 23:59. - **This is the final deadline for submitting the Homework assignment. If you miss this, you cannot continue the course and thus cannot earn a grade this semester !!!**

**Assessment:** Correctly solving the homework and having it accepted by the instructors is a prerequisite for taking the Coding test, and thus earning a grade on this course. The quality of the homework is not graded in detailed points; assessment is binary: pass / fail. A mostly correct and mostly complete ("pass") solution submitted by the first, normal deadline is worth 5 points. Solutions submitted following the first, normal deadline will not receive these 5 points.

**Requirement:** Wherever possible, we expect efficient, vectorized solutions. For further details, see Moodle, the "Homework assignment - Frequently Asked Questions" forum in module "Week 8", and the discussion topic "Can I use Python loops?".


Imports.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from dndmsc26autumn_hw1_tester import Tester

## **A**: Loading the dataset

**Information about the dataset**:
The original dataset comes from the UCI Machine Learning Repository (http://archive.ics.uci.edu/ml/datasets/Student+Performance), which have been slightly modified. It contains data about students studying in two Portuguese secondary schools, including their third-period and final grades (0-20). In this homework, **we will use neural networks to estimate the students' school performance based on their social and financial background.**

The testing script opens the dataset stored in text form and places it in the `content` string. Below, we print both its length and the first 500 characters of the string. Each line in the dataset file gives the values of the variables belonging to one sample. Within a line, the values corresponding to the individual variables are separated by semicolons. The first line of the dataset contains the variable names.

Your task is to **convert the `content` string into a NumPy array containing the values of the dataset variables.** The array data type must be `np.float32` (floating point). Store the variable values in the array `dataset_noisy` with shape `(n_samples, n_variables)`. We will separate the input variables and labels later.

The dataset contains missing values in some places: part of the data was not available when the dataset was recorded. More specifically, missing values occur in the columns describing the parents' occupations and the student's guardian. These are the ninth (`Mjob`), tenth (`Fjob`), and twelfth (`guardian`) columns. In the `content` string, missing values are absent between the corresponding semicolons, so in such cases two (or more) semicolons appear consecutively.

You must handle and detect the missing values while reading the data: the missing values must be represented by `np.nan` (Not-a-Number) values in the resulting `dataset_noisy` array.


In [ ]:
tester = Tester()
content = tester.get_dataset_content()

print("Number of characters in dataset:", len(content))
print(content[:500])


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('dataset_load', dataset_noisy)

## **B**: Handling missing data

We will not be able to continue working with NaN values, because most NumPy/PyTorch operations do not support them, and for any array that contains at least one NaN value, the output of these operations will also become NaN, including for example our neural network's prediction and loss. The latter would prevent us from optimizing the network using gradient-based methods. Therefore, we will eliminate the NaN values representing missing data in the following way:

**For each variable that may contain NaN values, create a new variable indicating whether a valid value is present for that variable in the given sample or not.** If a valid value is present, the new variable should take the value 1; in the NaN case, it should take the value 0. Place the original data and the new variables into the `dataset` array: the columns of the new variables must be appended **after** all columns containing the original variables, in the appropriate order! The previous task description mentioned which columns may contain NaN values.</br>
Since columns (variables) with indices #8, #9, and #11 in our 33-variable dataset may contain NaN values, the new `dataset` array will have 36 variables, and the three new variables will be placed in the last three columns. For example, in column #33, a row should contain 0 if the same row in column #8 contains NaN, and 1 otherwise. Similarly, create columns #34 and #35 based on columns #9 and #11.

Finally, **replace all NaN values in the resulting array with zeros!**


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('dataset_fill_missing', dataset)

## **C**: Splitting into training, validation, and test sets

**Randomly shuffle the samples** in the `dataset` array. In general, this is advisable because the samples in the dataset may be ordered according to some property. Without shuffling, the label distributions in the resulting splits could differ significantly.

After that, **split the array into training, validation, and test sets** in a 50%, 25%, 25% ratio. Name the three arrays containing these sets `dataset_split_train`, `dataset_split_val`, and `dataset_split_test`, respectively!


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('dataset_split', dataset_split_train, dataset_split_val, dataset_split_test)

## **D**: Creating data iterators for the regression task

Gradient-based training of neural networks is most often implemented using iterators that traverse the dataset and produce the input and label batches required for training. In the regression task, the target label variable to be estimated will be the students' final grades (the column named `G3` in the original dataset text file, i.e. the column with index 32), while the input variables will be all other attributes, including the three newly created variables.

In this subtask, **you need to create three iterable objects** named `dataloader_reg_train`, `dataloader_reg_val`, and `dataloader_reg_test`, which provide iterators that traverse the arrays containing the training, validation, and test sets created in the previous subtask. **At each step, the iterators must return two tensors**, storing one batch of input data and one batch of label data, respectively. Therefore, the shapes of the returned tensors will be `(batch_size, 35)` and `(batch_size, 1)`. The latter tensor (containing the labels) will store the students' final grades, while the former will store the values of all other dataset variables for the corresponding samples. `batch_size` denotes the number of samples that our neural network receives at once in a single batch, for example 32.

It is important to emphasize that the iterators should already return sample batches derived from the above NumPy arrays (`np.ndarray`) as `torch.tensor` objects (more precisely, as two-element tuples containing `torch.tensor` objects), since training neural networks defined in PyTorch definitely requires PyTorch tensors. The tensors must use the **data type** 4-byte floating point, i.e. `torch.float32`.

The iterators enumerating the **training and validation sets must traverse their samples in random order**. The iterator enumerating the test set must not change the order of the `dataset_split_test` array!

The implementation of the iterables can be solved in several ways. For example, by creating a custom generator/iterator, or by using and possibly subclassing the `Dataset` and `DataLoader` classes from the `torch.utils.data` module.


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('reg_iter', dataloader_reg_train, dataloader_reg_val, dataloader_reg_test)

## **E**: Defining the regression neural network

**Define the class implementing the neural network to be used for the regression task**, as a subclass of the general `torch.nn.Module` class, then instantiate it and assign the instance to the variable `reg_model`!

The regression neural network must **contain 3 fully connected layers with 25, 25, and 1 neurons, respectively, and apply the ReLU activation function between the layers!**

The new class must implement the member function `forward(self, x)`, which evaluates the neural network (hypothesis function) on the input tensor `x`. The `forward` function will receive the input tensors coming from the data iterator defined earlier and produce the label predictions from them. It is advisable to initialize the layers of the neural network in the constructor of the class.


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('reg_model_architecture', reg_model)

## **F**: Training the network for the regression task

**Train the** `reg_model` **neural network** on the training set using the iterator given by `dataloader_reg_train`, and use the iterator of `dataloader_reg_val` for validation. A single full pass through the training and validation datasets by the iterators (even in random order) defines one _epoch_.

Use the **mean squared error (MSE) loss**, which is commonly used for regression. **Measure the training and validation losses in every epoch, then after training, plot how these values changed during training on a shared plot.** To produce the plots, you may for example use the `matplotlib` library. Make sure that the differences between the curves remain clearly visible on the plot even near the end of training. If necessary, you may manually set which portion of the y-axis the plot displays. **The plot must include a legend** indicating which curve corresponds to the training loss and which to the validation loss.

You may determine the training duration (the number of epochs) manually based on the validation loss curve: try to choose it so that the neural network is trained well, but severe overfitting has not yet appeared.

For training, choose a gradient-based **optimization algorithm** from the `torch.optim` module (e.g. `SGD`, `Adam`, `RMSprop`, etc.), and also choose an appropriate **learning rate**.

After training, **measure the average loss on the full test set** and assign the resulting value to the variable `test_mse`!

After that, **select a few samples from the test set** (for example using the iterator of `dataloader_reg_test`), and **predict their labels** based on the values of the input variables, then **print the predicted and true labels for each sample**. This way, we can also see from examples how good the predictions of our neural network are.


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('reg_model_learning', test_mse)

## **G**: Creating data iterators for the classification task

Similarly to task D, here we also need to create three iterables which provide iterators over the three dataset splits. However, this time we will learn to solve a slightly different task, so the shapes / data types of the returned tensors will also differ. **Create** the **iterable objects** named `dataloader_cl_train`, `dataloader_cl_val`, and `dataloader_cl_test`, which should, again, provide iterators that traverse the arrays containing the training, validation, and test sets created in subtask C. In the classification task, we use all variables in the dataset as input variables except the students' three third-period grades (`G1`, `G2`, `G3`, the columns with indices #30, #31, and #32 in the original dataset text file). As labels, we define three categories and assign every student to one of them:
- **"Low-performing students"** category, category index #0: Students belong here if at least two of their grades are lower than 11.
- **"Moderately performing students"** category, category index #1: Students belong here if their average grade is lower than 14, but they do not belong to the "Low-performing students" category.
- **"High-performing students"** category, category index #2: Students belong here if they do not belong to either of the other two categories.

Based on the above, the tensors returned by the iterators will have shapes `(batch_size, 33)` and `(batch_size,)`. The latter tensor (containing the labels) will store the category indices of the students (that is, values 0, 1, or 2, with data type `torch.int64`), while the former will store the values of all used dataset variables for the corresponding samples with data type `torch.float32`. `batch_size` once again denotes the number of samples and must be set manually.

The iterators enumerating the **training and validation sets must traverse the corresponding samples in random order**. The iterator enumerating the test set must not change the order of the `dataset_split_test` array!


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('cl_iter', dataloader_cl_train, dataloader_cl_val, dataloader_cl_test)

## **H**: Defining the classification neural network

**Define the class implementing the neural network to be used for the classification task**, which, similarly to the class implementing the regression neural network, must be a subclass of the general `torch.nn.Module` class. Instantiate the new class and assign the instance to the variable `cl_model`!

The classification neural network must **contain 3 fully connected layers. The first two layers must contain 30 and 20 neurons, respectively, and apply the ReLU activation function after these layers.** The final layer must be designed for three-class classification!


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('cl_model_architecture', cl_model)

## **I**: Training the network for the multi-class classification task

**Train the** `cl_model` **neural network** on the training set using the iterator from `dataloader_cl_train`, and use the iterator from `dataloader_cl_val` for validation!

Use the **cross-entropy (CE) loss**, which is commonly used in multi-class classification. **Measure the training and validation losses in every epoch, then after training plot how these values changed during training on a shared plot.** On the plot, again make sure that there is a legend and that the curves are easy to distinguish.

This time, training must be stopped using the **early stopping** technique. If the validation loss does not improve for a given number of epochs (`patience`), training should end, and **the weights of `cl_model` must be restored from the epoch in which the validation loss was best!** This must happen automatically.

After training, **measure the average loss on the full test set** and assign the resulting value to the variable `test_ce`!

**Also compute the _accuracy_ metric on the test set samples** and assign the resulting value to the variable `test_acc`!! The _accuracy_ metric gives the proportion of correctly classified samples. For example, if while predicting the categories of 150 samples we correctly identify the right category in 75 cases, then the value of _accuracy_ is 0.5.

After that, **select a few samples from the test set** (for example using the iterator given by `dataloader_cl_test`), and **predict their labels** (their category indices) based on the values of the input variables, then **print the predicted and true labels for each sample** (the category indices). This way, we can also see from examples how good the predictions of our neural network are.

Finally, **compute the confusion matrix of the network on the test set and plot it**! The confusion matrix is a k x k matrix for k categories, where the element at index `[i,j]` gives how many samples belonging to category `i` in the dataset were predicted by our model as category `j`. Good model performance is indicated when as many elements as possible fall on the main diagonal of the confusion matrix. For the computation, you may also use an external library, but it can be done in NumPy as well. For plotting, you may for example use the `imshow` function from `matplotlib.pyplot`. Make sure that it is clearly visible on the plot which axis lists the true categories and which lists the predicted categories, that the category indices are visible at the ends of the rows and columns, and that the cells contain the corresponding sample counts!


In [ ]:
# implement your solution BELOW





# implement your solution ABOVE

tester.test('cl_model_learning', test_ce, test_acc)